# IMDB Sentiment Classification — Colab Training Notebook
Use this notebook to do all GPU-heavy work (Model V1 + V2 training, MLflow tracking) on Google Colab's free GPU.

**Workflow:** Clone your GitHub repo here -> train -> push trained `artifacts/` folder back to GitHub (or download it) -> use it locally on Windows for Docker/Kubernetes/API steps.

Steps:
1. Set Runtime -> Change runtime type -> GPU (T4)
2. Run the cells below in order

In [5]:
!git clone https://github.com/shaheer414/sentiment-project.git
%cd sentiment-project

Cloning into 'sentiment-project'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 30 (delta 3), reused 30 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 15.22 KiB | 247.00 KiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/sentiment-project


In [6]:
# 2. Install dependencies
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 124.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 126.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 95.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 105.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [8]:
# 3. Confirm GPU is available
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

RuntimeError: duplicate registrations for aten.linspace.Tensor_Tensor

In [ ]:
# 4. Train Model V1 (LSTM baseline)
!python training/train_v1.py

In [ ]:
# 5. Train Model V2 (fine-tuned DistilBERT) -- benefits most from GPU
!python training/train_v2.py

In [ ]:
# 6. Compare both models (writes artifacts/model_comparison_report.csv)
!python training/compare_models.py

In [ ]:
# 7. (Optional) View MLflow UI inside Colab using pyngrok
!pip install -q pyngrok
import subprocess, time
from pyngrok import ngrok

# Get a free token at https://dashboard.ngrok.com/get-started/your-authtoken
# ngrok.set_auth_token('YOUR_NGROK_TOKEN')

get_ipython().system_raw('mlflow ui --port 5000 &')
time.sleep(5)
public_url = ngrok.connect(5000)
print('MLflow UI:', public_url)

In [ ]:
# 8. Zip artifacts + mlruns to download locally for use on Windows
!zip -r trained_artifacts.zip artifacts mlruns
from google.colab import files
files.download('trained_artifacts.zip')

In [ ]:
# 9. (Optional) Commit and push trained artifacts back to GitHub
# Note: large model weights should normally use Git LFS or be excluded via .gitignore
# and re-downloaded/retrained instead of committed directly.
!git config --global user.email "you@example.com"
!git config --global user.name "Your Name"
# !git add artifacts/model_comparison_report.csv
# !git commit -m "Add training results from Colab"
# !git push